1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score

2. Load Dataset

In [ ]:
df = pd.read_csv(r"C:\itsChutiii\Code & Cry (Uni)\Bsc.Hons Data science\Y02 SEM 01\Introduction to Data science\Data sets\heart.csv")

df.head()

3. Dataset Overview

In [ ]:
print(df.shape)

print("\nData Info:")
print(df.info())

print("\nStatistical Summary:")
print(df.describe())

4. Missing Values & Duplicates

In [ ]:
print("\nMissing values in each column:")
print(df.isnull().sum())

print("\nDuplicate Rows:")
print(df.duplicated().sum())

# Remove duplicates
df = df.drop_duplicates()

5. Outlier Detection & Capping (IQR Method)

In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns

if 'target' in numeric_cols:
    numeric_cols = numeric_cols.drop('target')


def cap_outliers_iqr(df_temp, col):
    Q1 = df_temp[col].quantile(0.25)
    Q3 = df_temp[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    mask_lower = df_temp[col] < lower
    mask_upper = df_temp[col] > upper

    df_temp.loc[mask_lower, col] = lower
    df_temp.loc[mask_upper, col] = upper


for col in numeric_cols:
    cap_outliers_iqr(df, col)

6. Data Visualization

In [ ]:
sns.countplot(x='target', data=df)
plt.title("Heart Disease Distribution")
plt.show()

sns.countplot(x='sex', hue='target', data=df)
plt.title("Gender vs Heart Disease")
plt.show()

sns.histplot(df['age'], bins=20, kde=True)
plt.title("Age Distribution")
plt.show()

sns.boxplot(x=df['chol'])
plt.title("Cholesterol Outliers")
plt.show()

plt.figure(figsize=(12,8))
sns.heatmap(df.corr(), annot=True, cmap="coolwarm")
plt.title("Correlation Heatmap")
plt.show()


7. Feature Scaling (Normalization & Standardization)

In [ ]:
scaler = MinMaxScaler()

df_normalized = pd.DataFrame(
    scaler.fit_transform(df),
    columns=df.columns
)

df_normalized.head()


standard = StandardScaler()

df_standardized = pd.DataFrame(
    standard.fit_transform(df),
    columns=df.columns
)

df_standardized.head()

8. Splitting Dataset

In [ ]:
X = df.drop("target", axis=1)
y = df["target"]

print(X.head())
print(y.head())

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

9. Model Training (Random Forest)

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf_model.fit(X_train, y_train)

print("Model Training Completed")

10. Predictions

In [ ]:
y_pred = rf_model.predict(X_test)

results = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted": y_pred
})

results.head(10)

11. Model Evaluation

In [ ]:
accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)
print("\nConfusion Matrix:\n", cm)

f1 = f1_score(y_test, y_pred)
print("F1 Score:", f1)

12. Confusion Matrix Visualization

In [ ]:
plt.figure(figsize=(6,5))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["No Disease", "Disease"],
    yticklabels=["No Disease", "Disease"]
)

plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

13. Feature Importance

In [ ]:
feature_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf_model.feature_importances_
})

feature_importance = feature_importance.sort_values(by="Importance", ascending=False)

feature_importance

Feature Importance Plot

In [ ]:
plt.figure(figsize=(10,6))

sns.barplot(
    data=feature_importance,
    x="Importance",
    y="Feature"
)

plt.title("Feature Importance")
plt.show()